# CodeAct Tools and Live Objects

Notebook 2 taught the decision rule: Predict for one-shot structured judgment, CodeAct for work that needs Python, tools, state, or iteration.

This notebook stays on the CodeAct side. The question now is an API-design question: if the model is going to write Python against your agent object, what should that object expose?

We'll scale the bookshop from a tiny shelf to the Project Gutenberg catalog. The catalog is large enough that dumping it into the prompt would be wasteful and brittle. The right pattern is to keep large state in Python and expose small deterministic helper methods over it.


## Prerequisites

Install NOOA from GitHub with [uv](https://docs.astral.sh/uv/):

```bash
uv add "nooa @ git+https://github.com/NVIDIA-NeMo/labs-OO-Agents.git@main"
```

The setup cell below lists several providers. Uncomment the one you want to use. For hosted providers, replace `"your-api-key"` with a real key. NOOA is also compatible with **local inference** (Ollama, vLLM, any OpenAI-compatible endpoint) - those need no API key, just an `api_base`.


## Setup

NOOA works with any LiteLLM-supported model - hosted or local. Pick one below before running the rest of the notebook.


In [ ]:

from nooa.unifiedllm.registry import get_llm_client

# NOOA works with any LiteLLM-supported model - hosted or local.
# Pick one below. Replace "your-api-key" with a real key for hosted providers;
# local providers (Ollama, vLLM) do not need a key, just pass api_base.

model = get_llm_client("claude-haiku-4-5", api_key="your-api-key")                                        # Anthropic
# model = get_llm_client("gpt-5-mini", api_key="your-api-key")                                              # OpenAI
# model = get_llm_client("ollama_chat/qwen3:1.7b", api_base="http://localhost:11434")                       # Ollama (local, no key)
# model = get_llm_client("hosted_vllm/Qwen/Qwen3-1.7B", api_base="http://localhost:8000/v1")                # vLLM (local, no key)
# model = get_llm_client("openai/nvidia/openai/gpt-oss-20b", api_key="your-api-key", api_base="https://inference-api.nvidia.com/v1")  # NVIDIA hosted (OpenAI-compatible)


## Load A Real Catalog

Project Gutenberg publishes a CSV of its public-domain catalog. It has tens of thousands of rows, with title, author, language, subjects, and bookshelves.


In [ ]:

import pandas as pd
from typing import Literal

from pydantic import BaseModel, Field

from nooa import Agent, print_prompt, strategy
from nooa.agentdoc import doc
from nooa.strategies import PredictStrategy


catalog = pd.read_csv("https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv")

catalog = catalog.rename(
    columns={
        "Text#": "id",
        "Title": "title",
        "Authors": "author",
        "Language": "language",
        "Subjects": "topics",
        "Bookshelves": "bookshelves",
    }
)[["id", "title", "author", "language", "topics", "bookshelves"]]

csv_chars = len(catalog.to_csv(index=False))

print(f"Catalog rows: {len(catalog):,}")
print(f"In-memory footprint: {catalog.memory_usage(deep=True).sum():,} bytes")
print(f"Serialized as CSV:   ~{csv_chars // 4:,} tokens, very roughly")
catalog.head(3)


## The Wrong Surface: Raw State As The Interface

A DataFrame is a fine Python object, but it is not a good agent API by itself. If the model only sees "there is a catalog somewhere", it has to rediscover column names, pandas idioms, filtering rules, and return shapes every time.

The fix is the same as normal software design: keep the raw state private, then expose a small public interface with methods that do deterministic work.


## The Right Surface: Helpers Over Hidden State

The catalog below lives on `_catalog`, so it is hidden from `doc(self)` by Python's underscore convention. The LLM does not need the DataFrame directly. It needs good methods: search by title, search by topic, filter by language, and look up authors.


In [ ]:

class Mood(BaseModel):
    vibe: Literal["curious", "lost", "hostile", "returning_a_book", "just_browsing"]
    confidence: float = Field(ge=0, le=1)
    note: str


class Recommendation(BaseModel):
    title: str
    author: str
    language: str
    why_this_book: str


class BookshopAgent(Agent, llm=model):
    """You run a used bookshop and recommend public-domain books."""

    def __init__(self, catalog: pd.DataFrame):
        super().__init__()
        self._catalog = catalog

    def search_titles(self, query: str, n: int = 10) -> list[dict]:
        """Return up to n books whose title contains query, case-insensitive."""
        q = query.lower()
        mask = self._catalog["title"].str.lower().str.contains(q, na=False, regex=False)
        return self._catalog[mask].head(n).to_dict(orient="records")

    def by_topic(self, topic: str, n: int = 10) -> list[dict]:
        """Return up to n books whose topics or bookshelves mention topic."""
        q = topic.lower()
        text = self._catalog["topics"].fillna("") + " " + self._catalog["bookshelves"].fillna("")
        mask = text.str.lower().str.contains(q, na=False, regex=False)
        return self._catalog[mask].head(n).to_dict(orient="records")

    def in_language(self, language_code: str, n: int = 10) -> list[dict]:
        """Return up to n books in the given ISO 639-1 language code, such as en, fr, or de."""
        mask = self._catalog["language"].str.lower() == language_code.lower()
        return self._catalog[mask].head(n).to_dict(orient="records")

    def by_author(self, author: str, n: int = 10) -> list[dict]:
        """Return up to n books whose author field contains author."""
        q = author.lower()
        mask = self._catalog["author"].str.lower().str.contains(q, na=False, regex=False)
        return self._catalog[mask].head(n).to_dict(orient="records")

    @strategy(PredictStrategy())
    async def read_the_customer(self, opening_line: str) -> Mood:
        """Classify the customer's vibe from their opening line."""
        ...

    async def recommend(self, customer_wants: str) -> Recommendation:
        """Find one public-domain book for this customer.

        Use self.search_titles, self.by_topic, self.in_language, and self.by_author
        to explore the catalog. Do not ask for or print the whole catalog. Return
        one recommendation with a short, concrete reason.
        """
        ...

    async def recommend_for_customer(self, opening_line: str) -> Recommendation:
        """Read the customer's vibe, then recommend one public-domain book.

        Start by awaiting self.read_the_customer. Then use the catalog helper
        methods to find a book whose subject and tone fit the customer's mood.
        """
        ...


agent = BookshopAgent(catalog)
print(doc(agent))


Look at the rendered docs. The public helper methods are visible. `_catalog` is not. That is the surface you designed for the model.


## The Catalog Is Not Prompt Text

Arguments are rendered by the framework, and public agent API documentation is rendered by `doc(self)`. But the hidden DataFrame itself is not serialized into the prompt. The model can only reach it through the helper methods you exposed.


In [ ]:

would_be_chars = len(repr(catalog.to_dict(orient="records")))
print(f"If serialized directly, the catalog would be roughly {would_be_chars // 4:,} tokens.")

await print_prompt(agent.recommend, customer_wants="a short English mystery, nothing too heavy")


The prompt should show the method signatures and docstrings, not the tens of thousands of catalog rows. That is the pass-by-reference pattern: large state stays in Python, and the LLM manipulates it through a small API.


## Deterministic Helpers Are Still Useful Outside The LLM

The helper methods are normal Python. You can test them, call them in notebooks, and refactor them without touching prompt glue.


In [ ]:

hits = agent.by_topic("mystery", n=20)
short_titles = [book for book in hits if len(str(book["title"])) < 45]

print(f"Mystery hits: {len(hits)}")
print(f"Short title candidates: {len(short_titles)}")
short_titles[:3]


## Now Let CodeAct Use The Surface

The generation method can build the same kind of intermediate state inside its REPL: call helpers, assign variables, filter lists, inspect candidates, then return a structured `Recommendation`.


In [ ]:

rec = await agent.recommend("a short English mystery, nothing too heavy")
print(f"Ada recommends: {rec.title} by {rec.author} ({rec.language})")
print(f"Why: {rec.why_this_book}")


Open the trace viewer if it is not already running:

```bash
nooa start-dev
```

Re-run the recommendation cell and inspect the CodeAct trace. You should see generated Python cells, calls to helper methods, intermediate values, and a final structured return.


## CodeAct Is A REPL, Not Stateless Tool Calling

In a stateless tool-call loop, every tool call returns text into the message log. In CodeAct, tool calls are Python expressions inside a persistent execution environment. Variables assigned in one generated cell can still be used in the next generated cell.

For example, the model can do this across CodeAct turns:

```python
hits = self.by_topic("mystery", n=50)
short_titles = [b for b in hits if len(b["title"]) < 45]
```

Then later:

```python
best = short_titles[0]
return_result({"title": best["title"], ...})
```

That persistence is why a clean helper API matters. The model can build small working sets while the large object stays where it belongs: in Python memory.


## Generation Methods Can Be Part Of The Surface

`read_the_customer` is a Predict method, but it is still a method on `self`. A CodeAct method can call it as one step in a larger workflow.


In [ ]:

rec = await agent.recommend_for_customer(
    "I suppose I need something clever, unless all you have is nonsense."
)
print(f"Recommended: {rec.title} by {rec.author} ({rec.language})")
print(f"Why: {rec.why_this_book}")


## Next: Controlling Visibility

This notebook deliberately shaped what the model could see: public helper methods were visible, private raw state was hidden. Notebook 4 makes those visibility rules explicit: `doc()`, `@hidden`, leading underscores, `Annotated[T, hidden]`, `@spec(hidden=False)`, and `with hidden:`.


## Recap

- CodeAct works best when the agent exposes a small, useful Python API.
- Large state should usually stay private or hidden.
- Deterministic helpers are better than asking the model to rediscover pandas logic.
- The LLM calls public methods as tools because they are part of the object surface.
- CodeAct keeps a persistent Python environment, so intermediate variables can survive across generated cells.


## Exercises

1. Add `recent_books(n: int = 10) -> list[dict]` using the Gutenberg id as a rough proxy for recency, then ask for a recent-feeling recommendation.
2. Add `count_by_author(author: str) -> int` and use it to prefer prolific authors when the request is vague.
3. Hide one public helper with `@hidden`, print `doc(agent)`, and confirm it disappears.
4. Add a helper that returns only `title`, `author`, and `language` so tool results stay smaller.
